In [26]:
print ("Hello word")

Hello word



# TAHAP 1: CUSTOM EXCEPTIONS (ERROR HANDLING)


In [27]:


class ReservationError(Exception):
    """Base class exception untuk semua error terkait reservasi di IS Lab."""
    def __init__(self, message: str):
        super().__init__(message)
        self.message = message

class CapacityError(ReservationError):
    """Dilemparkan ketika jumlah peserta melebihi kapasitas Meeting Room."""
    pass

class OverlapError(ReservationError):
    """Dilemparkan ketika sistem mendeteksi adanya double booking."""
    pass

class ValidationError(ReservationError):
    """Dilemparkan ketika terdapat input data yang tidak masuk akal."""
    pass


# TAHAP 2: ENUM & ENTITAS DASAR


In [28]:
from enum import Enum

# 1. Status Reservasi
class ReservationStatus(Enum):
    """Status minimum untuk sebuah reservasi di IS Lab."""
    ACTIVE = "ACTIVE"
    CANCELLED = "CANCELLED"

In [29]:
# 2. Entitas Member
class Member:
    """Representasi anggota laboratorium."""
    def __init__(self, member_id: str, name: str, role: str):
        self.member_id = member_id
        self.name = name
        self.role = role

    def __str__(self) -> str:
        return f"[{self.member_id}] {self.name} - {self.role}"

In [30]:
# 3. Entitas Resource (Parent Class)
class Resource:
    """Base class untuk semua fasilitas laboratorium."""
    def __init__(self, resource_id: str, name: str):
        self.resource_id = resource_id
        self.name = name

    def __str__(self) -> str:
        return f"[{self.resource_id}] {self.name}"

In [31]:
# 4. Entitas Workstation (Child Class)
class Workstation(Resource):
    """Fasilitas dengan spesifikasi GPU. Turunan dari Resource."""
    def __init__(self, resource_id: str, name: str, gpu_spec: str):
        super().__init__(resource_id, name)
        self.gpu_spec = gpu_spec

    def __str__(self) -> str:
        return f"{super().__str__()} (GPU: {self.gpu_spec})"

In [32]:
# 5. Entitas Meeting Room (Child Class)
class MeetingRoom(Resource):
    """Fasilitas dengan kapasitas maksimal. Turunan dari Resource."""
    def __init__(self, resource_id: str, name: str, capacity: int):
        super().__init__(resource_id, name)
        self.capacity = capacity

    def __str__(self) -> str:
        return f"{super().__str__()} (Capacity: {self.capacity} pax)"


# TAHAP 3: CLASS TRANSAKSI (RESERVATION)


In [33]:
from datetime import datetime, timedelta

class Reservation:
    """Merekam data reservasi yang dilakukan oleh Member pada sebuah Resource."""
    
    def __init__(self, reservation_id: str, member: Member, resource: Resource, start_time: datetime, duration_hours: int):
        self.reservation_id = reservation_id
        self.member = member
        self.resource = resource
        self.start_time = start_time
        self.duration_hours = duration_hours
        
        # Sesuai requirement, status awal selalu ACTIVE
        self.status = ReservationStatus.ACTIVE 

    @property
    def end_time(self) -> datetime:
        """Menghitung waktu selesai secara otomatis berdasarkan waktu mulai dan durasi."""
        return self.start_time + timedelta(hours=self.duration_hours)

    def cancel(self):
        """Membatalkan reservasi dengan menerapkan asumsi kebijakan waktu."""
        # 1. Cek apakah sudah dibatalkan sebelumnya
        if self.status == ReservationStatus.CANCELLED:
            raise ValidationError("Reservasi sudah berstatus CANCELLED dan tidak dapat dibatalkan kembali.")

        # (Dinonaktifkan sementara agar Skenario E dari dokumen tidak ditolak oleh validasi waktu nyata)    
        # 2. Asumsi Kebijakan: Tidak bisa membatalkan jika jadwal sudah terlewati
        # if datetime.now() > self.start_time:
        #     raise ValidationError("Tidak dapat membatalkan reservasi yang jadwalnya sudah terlewat.")
            
        self.status = ReservationStatus.CANCELLED

    def __str__(self) -> str:
        # Mengubah format waktu komputer menjadi format yang mudah dibaca (Contoh: 10 Sep 2026, 09:00)
        waktu_format = self.start_time.strftime("%d %b %Y, %H:%M")
        return f"Res[{self.reservation_id}] {self.member.name} -> {self.resource.name} ({waktu_format} selama {self.duration_hours} jam) | Status: {self.status.value}"


# TAHAP 4: CLASS MANAJER (SYSTEM CONTROLLER)


In [34]:
class ISLabReservationSystem:
    """Sistem sentral untuk mengelola anggota, fasilitas, dan reservasi IS Lab."""
    
    def __init__(self):
        # Menggunakan dictionary untuk pencarian cepat berdasarkan ID
        self.members = {}       
        self.resources = {}     
        # Menggunakan list karena kita akan banyak melakukan iterasi/perulangan waktu
        self.reservations = []  
        
        # Counter otomatis untuk membuat Reservation ID (Contoh: REV-001)
        self._reservation_counter = 1

    def add_member(self, member: Member):
        """Menambahkan anggota baru ke dalam sistem."""
        if member.member_id in self.members:
            raise ValidationError(f"Member dengan ID {member.member_id} sudah terdaftar.")
        self.members[member.member_id] = member
        
    def add_resource(self, resource: Resource):
        """Menambahkan fasilitas baru ke dalam sistem."""
        if resource.resource_id in self.resources:
            raise ValidationError(f"Resource dengan ID {resource.resource_id} sudah terdaftar.")
        self.resources[resource.resource_id] = resource

    def make_reservation(self, member_id: str, resource_id: str, start_time: datetime, duration_hours: int, participants: int = 1) -> Reservation:
        """Fungsi utama untuk membuat dan memvalidasi reservasi."""
        
        # 1. Pengecekan Eksistensi Entitas
        member = self.members.get(member_id)
        if not member:
            raise ValidationError(f"Member ID {member_id} tidak ditemukan.")
            
        resource = self.resources.get(resource_id)
        if not resource:
            raise ValidationError(f"Resource ID {resource_id} tidak ditemukan.")

        # 2. Validasi Durasi & Waktu Lampau (Asumsi Kebijakan)
        if duration_hours <= 0:
            raise ValidationError("Durasi peminjaman harus lebih besar dari 0.")

        # (Dinonaktifkan  agar Skenario Wajib A 10 Sept 09:00 dari dokumen tidak terblokir oleh waktu nyata)
        # if start_time < datetime.now():
        #     raise ValidationError("Tidak dapat memesan fasilitas untuk waktu di masa lalu.")

        # 3. Validasi Kapasitas Ruangan
        if isinstance(resource, MeetingRoom):
            if participants > resource.capacity:
                raise CapacityError(f"Peserta ({participants}) melebihi kapasitas {resource.name} ({resource.capacity} pax).")

        # 4. Validasi Double Booking (Menggunakan Logika "Jadwal Aman" Buatanmu)
        new_end_time = start_time + timedelta(hours=duration_hours)
        
        for res in self.reservations:
            # Kita hanya mengecek resource yang sama dan status yang masih ACTIVE
            if res.resource.resource_id == resource_id and res.status == ReservationStatus.ACTIVE:
                
                # Kasus 2: Selesai duluan sebelum pesanan lama mulai
                aman_selesai_duluan = new_end_time <= res.start_time
                # Kasus 1: Mulai belakangan setelah pesanan lama selesai
                aman_mulai_belakangan = start_time >= res.end_time
                
                # Jika TIDAK selesai duluan DAN TIDAK mulai belakangan, berarti posisi di tengah (Bentrok!)
                if not (aman_selesai_duluan or aman_mulai_belakangan):
                    raise OverlapError(f"Gagal! Jadwal bentrok dengan reservasi {res.reservation_id} ({res.start_time.strftime('%H:%M')} - {res.end_time.strftime('%H:%M')}).")

        # 5. Persetujuan & Pencatatan Akhir
        res_id = f"REV-{self._reservation_counter:03d}"
        self._reservation_counter += 1
        
        new_reservation = Reservation(res_id, member, resource, start_time, duration_hours)
        self.reservations.append(new_reservation)
        
        return new_reservation

    def view_reservations(self, member_id: str = None, resource_id: str = None):
        """Req 8: Menampilkan reservasi dengan opsi filter berdasarkan anggota atau resource."""
        print("\n--- Daftar Reservasi ---")
        ditemukan = False
        
        for res in self.reservations:
            # Jika filter member_id diisi, lewati data yang tidak cocok
            if member_id and res.member.member_id != member_id:
                continue
                
            # Jika filter resource_id diisi, lewati data yang tidak cocok
            if resource_id and res.resource.resource_id != resource_id:
                continue
                
            print(res)
            ditemukan = True
            
        if not ditemukan:
            print("Tidak ada reservasi yang cocok dengan pencarian.")

    def check_availability(self, check_time: datetime):
        """Req 9: Menampilkan resource yang tersedia pada satu titik waktu tertentu."""
        print(f"\n--- Ketersediaan Fasilitas pada {check_time.strftime('%d %b %Y, %H:%M')} ---")
        tersedia = []
        
        for res_id, resource in self.resources.items():
            is_available = True
            
            for res in self.reservations:
                # Kita cek khusus jadwal yang aktif dan ada pada resource ini
                if res.resource.resource_id == res_id and res.status == ReservationStatus.ACTIVE:
                    # Sebuah ruangan dianggap TIDAK TERSEDIA jika waktu pencarian 
                    # berada di antara waktu mulai (inklusif) dan waktu selesai (eksklusif)
                    if res.start_time <= check_time < res.end_time:
                        is_available = False
                        break
            
            if is_available:
                tersedia.append(resource)
                print(f" {resource}")
                
        if not tersedia:
            print("❌ Semua fasilitas penuh pada waktu tersebut.")


# TAHAP 5: PENGUJIAN SKENARIO WAJIB
<font size="2">Ini teks ukuran kecil</font>
<font size="5">Ini teks ukuran besar</font>

In [35]:
from datetime import datetime

# 1. Inisialisasi Sistem dan Data Awal
system = ISLabReservationSystem()

# Mendaftarkan Member
andi = Member("M001", "Andi", "Student")
sarah = Member("M002", "Sarah", "Assistant")
budi = Member("M003", "Budi", "Student")
system.add_member(andi)
system.add_member(sarah)
system.add_member(budi)

# Mendaftarkan Resource
ws1 = Workstation("WS01", "AI Workstation 1", "RTX 4090")
mr1 = MeetingRoom("MR01", "Discussion Room", 8)
system.add_resource(ws1)
system.add_resource(mr1)

print("--- HASIL PENGUJIAN SKENARIO WAJIB ---")

# Skenario A: Andi memesan AI Workstation 1 (10 Sept 2026, 09:00, 2 jam)
try:
    time_a = datetime(2026, 9, 10, 9, 0)
    res_a = system.make_reservation("M001", "WS01", time_a, 2)
    print(f"Skenario A (Seharusnya Berhasil): {res_a}")
except ReservationError as e:
    print(f"Skenario A Gagal: {e}")

# Skenario B: Sarah mencoba memesan AI Workstation 1 (10 Sept 2026, 10:00, 2 jam) - Overlap
try:
    time_b = datetime(2026, 9, 10, 10, 0)
    res_b = system.make_reservation("M002", "WS01", time_b, 2)
    print(f"Skenario B Berhasil: {res_b}")
except ReservationError as e:
    print(f"Skenario B (Seharusnya Ditolak): {e}")

# Skenario C: Sarah memesan AI Workstation 1 (10 Sept 2026, 11:00, 1 jam)
try:
    time_c = datetime(2026, 9, 10, 11, 0)
    res_c = system.make_reservation("M002", "WS01", time_c, 1)
    print(f"Skenario C (Seharusnya Berhasil): {res_c}")
except ReservationError as e:
    print(f"Skenario C Gagal: {e}")

# Skenario D: Budi memesan Discussion Room (11 Sept 2026, 13:00, 2 jam, 10 peserta) - Kapasitas
try:
    time_d = datetime(2026, 9, 11, 13, 0)
    res_d = system.make_reservation("M003", "MR01", time_d, 2, participants=10)
    print(f"Skenario D Berhasil: {res_d}")
except ReservationError as e:
    print(f"Skenario D (Seharusnya Ditolak): {e}")

# Skenario E: Andi membatalkan reservasi pertamanya
try:
    res_a.cancel()
    print(f"Skenario E (Seharusnya Batal): Reservasi Andi berhasil dibatalkan. Status saat ini: {res_a.status.value}")
    
    # PEMBUKTIAN SLOT KEMBALI TERSEDIA
    res_bukti = system.make_reservation("M002", "WS01", time_a, 2)
    print(f"Bukti Slot Tersedia: Sarah kini berhasil memesan slot tersebut -> {res_bukti}")
    
except ReservationError as e:
    print(f"Skenario E Gagal: {e}")

# Uji Req 8: Lihat jadwal yang difilter khusus untuk Andi (M001)
system.view_reservations(member_id="M001")
# 1. Menguji "Menampilkan seluruh reservasi"
print("\n[UJI COBA 1: Tampilkan Semua]")
system.view_reservations()

# 2. Menguji "Filter berdasarkan anggota (Andi)"
print("\n[UJI COBA 2: Filter Anggota M001]")
system.view_reservations(member_id="M001")

# 3. Menguji "Filter berdasarkan resource (Workstation 1)"
print("\n[UJI COBA 3: Filter Resource WS01]")
system.view_reservations(resource_id="WS01")

# Uji Req 9: Cek ruangan kosong pada 10 Sept 2026 jam 11:30
waktu_cek = datetime(2026, 9, 10, 11, 30)
system.check_availability(waktu_cek)

--- HASIL PENGUJIAN SKENARIO WAJIB ---
Skenario A (Seharusnya Berhasil): Res[REV-001] Andi -> AI Workstation 1 (10 Sep 2026, 09:00 selama 2 jam) | Status: ACTIVE
Skenario B (Seharusnya Ditolak): Gagal! Jadwal bentrok dengan reservasi REV-001 (09:00 - 11:00).
Skenario C (Seharusnya Berhasil): Res[REV-002] Sarah -> AI Workstation 1 (10 Sep 2026, 11:00 selama 1 jam) | Status: ACTIVE
Skenario D (Seharusnya Ditolak): Peserta (10) melebihi kapasitas Discussion Room (8 pax).
Skenario E (Seharusnya Batal): Reservasi Andi berhasil dibatalkan. Status saat ini: CANCELLED
Bukti Slot Tersedia: Sarah kini berhasil memesan slot tersebut -> Res[REV-003] Sarah -> AI Workstation 1 (10 Sep 2026, 09:00 selama 2 jam) | Status: ACTIVE

--- Daftar Reservasi ---
Res[REV-001] Andi -> AI Workstation 1 (10 Sep 2026, 09:00 selama 2 jam) | Status: CANCELLED

[UJI COBA 1: Tampilkan Semua]

--- Daftar Reservasi ---
Res[REV-001] Andi -> AI Workstation 1 (10 Sep 2026, 09:00 selama 2 jam) | Status: CANCELLED
Res[REV-00


# TAHAP 6: EDGE CASE TESTING


In [36]:


print("\n--- HASIL PENGUJIAN EDGE CASE ---")

# Edge Case 1: Durasi Negatif atau Nol
try:
    time_ec1 = datetime(2026, 9, 12, 10, 0)
    system.make_reservation("M001", "WS01", time_ec1, 0)
except ReservationError as e:
    print(f"Edge Case 1 Berhasil Ditolak (Durasi 0): {e}")

# Edge Case 2: Membatalkan Reservasi yang Sudah Dibatalkan (Double Cancel)
try:
    res_a.cancel() # Percobaan batal kedua kali (karena sudah dibatalkan di Skenario E)
except ReservationError as e:
    print(f"Edge Case 2 Berhasil Ditolak (Double Cancel): {e}")

# Edge Case 3: ID Resource Tidak Ditemukan
try:
    time_ec3 = datetime(2026, 9, 12, 11, 0)
    system.make_reservation("M001", "GHOSTRoom", time_ec3, 2)
except ReservationError as e:
    print(f"Edge Case 3 Berhasil Ditolak (Resource Gaib): {e}")


--- HASIL PENGUJIAN EDGE CASE ---
Edge Case 1 Berhasil Ditolak (Durasi 0): Durasi peminjaman harus lebih besar dari 0.
Edge Case 2 Berhasil Ditolak (Double Cancel): Reservasi sudah berstatus CANCELLED dan tidak dapat dibatalkan kembali.
Edge Case 3 Berhasil Ditolak (Resource Gaib): Resource ID GHOSTRoom tidak ditemukan.
